# 03 — Pelatihan Model Utama (Tahap 2)

**Peran:** R2 (AI Model Engineer) · **Dataset:** `v4_seed1000_700trips.parquet`

Notebook ini menceritakan bagaimana model utama dibangun: arsitekturnya, tiga masalah serius yang ditemukan dan diperbaiki di sepanjang jalan, serta dua ablation study yang menentukan bentuk akhirnya.

Kode sesungguhnya berada di `ml/model.py` dan `ml/finetune.py`. Notebook ini menjelaskan, bukan menggandakan.

```bash
python -m ml.preprocess.build_windows   # jendela geser dari dataset v4
python -m ml.finetune                    # latih model (~35 menit CPU)
```

## 1. Bentuk data masukan

Model tidak menerima satu baris telemetri, melainkan **jendela 60 menit berturut-turut**. Alasannya: suhu punya inersia. Kondisi menit ke-60 sangat dipengaruhi apa yang terjadi di menit-menit sebelumnya, dan pintu yang terbuka di menit ke-10 baru terasa dampaknya beberapa menit kemudian.

Jendela digeser satu menit demi satu menit, **tidak pernah melintasi batas `trip_id`** — dua perjalanan berbeda tidak boleh disambung, karena menit terakhir trip A tidak berhubungan dengan menit pertama trip B.

Target forecast diambil dari **setelah** ujung jendela: bila jendela berisi menit 0–59, maka menit ke-59 adalah "sekarang", dan sasarannya adalah suhu pada menit ke-74, ke-89, dan ke-119. Jendela yang tidak punya cukup ruang di belakangnya dibuang, begitu pula jendela yang mengandung nilai kosong akibat packet loss.

In [ ]:
import numpy as np

for nama in ["train", "val", "test"]:
    d = np.load(f"../../data/processed/windows/windows_{nama}.npz")
    breach = (d["y_ttb"] != 999).mean()
    print(f"{nama:5s} : {d['X'].shape[0]:>7,} jendela  |  {breach*100:4.1f}% menuju breach")

## 2. Arsitektur

```
Jendela 60 x 12 ──> [GRU 2 lapis, hidden 64, dropout 0,2] ──> 64 angka ──┐
        │                                                                 ├──> 136 angka
        └────────> [6 statistik x 12 fitur, tanpa parameter] ──> 72 angka ┘        │
                                                                                    ├──> suhu t+15/30/60
                                                                                    ├──> 7 kelas mode kegagalan
                                                                                    └──> time-to-breach
```

**GRU** membaca jendela menit demi menit sambil memperbarui satu ringkasan internal 64 angka — inilah yang menangkap pola *urutan waktu*.

**Jalur statistik ringkasan** (rata-rata, simpangan baku, min, maks, nilai terakhir, tren) dihitung langsung dari jendela yang sama, tanpa parameter yang perlu dilatih. Jalur ini ditambahkan belakangan — alasannya dijelaskan di bagian 3.

Total **41.443 parameter**, sesuai batasan "puluhan ribu, bukan jutaan" agar inferensi CPU tetap cepat.

In [ ]:
import sys; sys.path.insert(0, "../..")
from ml.model import ColdTrackGRU

model = ColdTrackGRU()
n = lambda m: sum(p.numel() for p in m.parameters())
print(f"backbone GRU   : {n(model.gru):>7,}")
for nama, head in [("head forecast", model.head_forecast),
                   ("head failure", model.head_failure),
                   ("head TTB", model.head_ttb)]:
    print(f"{nama:15s}: {n(head):>7,}")
print(f"{'TOTAL':15s}: {n(model):>7,}")

## 3. Tiga masalah yang ditemukan dan diperbaiki

Model versi pertama gagal total: hanya 2 dari 7 kelas yang pernah ditebak, dan head TTB macet pada satu angka konstan. Tiga perbaikan berikut mengubahnya.

### 3.1 Skala fitur timpang 4500:1

| Fitur | Simpangan baku |
|---|---|
| `solar_radiation` | 270,0 |
| `reefer_duration_min` | 89,1 |
| `delta_temp` | 0,10 |
| `door_open` | 0,06 |

GRU menerima kedua belas angka mentah bersamaan, sehingga `solar_radiation` **menenggelamkan** `door_open` — padahal justru dua fitur terakhir yang paling menandakan anomali. Normalisasi setiap kolom ke rata-rata 0 dan simpangan baku 1 membuat semuanya bersuara setara.

### 3.2 Head forecast hanya kebagian 2,3% perhatian

Bobot loss awal `1,0 / 1,0 / 0,8` mengandaikan ketiga nilai loss sebanding besarnya. Kenyataannya tidak: setelah normalisasi, MAE forecast bernilai ~0,036 sementara CrossEntropy ~1,42 — sekitar 40 kali lebih besar.

| Tugas | Kontribusi ke loss total |
|---|---|
| forecast | 2,3% |
| klasifikasi | **90,4%** |
| time-to-breach | 7,3% |

Model praktis hanya belajar klasifikasi. Bobot disetel ulang menjadi `30 / 1,0 / 8,0` agar kontribusinya sebanding — forecast langsung membaik 37%.

### 3.3 Sentinel TTB tidak di-mask

Sekitar 80% baris memiliki `time_to_breach = 999`, penanda "tidak akan breach". Model dipaksa menghafal angka 999 untuk mayoritas data, dan akibatnya macet pada tebakan hampir konstan. Setelah baris bersentinel dikeluarkan dari perhitungan loss, head TTB mulai benar-benar belajar.

**Konsekuensi yang harus diingat:** karena kondisi sehat tidak pernah dilatih, keluaran head TTB untuk kondisi sehat **tidak terdefinisi**. Angka TTB wajib disembunyikan bila klasifikasi menunjuk kelas `A0`.

## 4. Ablation A — multi-task vs single-task

Spesifikasi awal menetapkan satu backbone melayani tiga head sekaligus. Ablation menguji apakah kompromi itu merugikan.

| Tugas | Multi-task | Single-task |
|---|---|---|
| Forecast t+30 | 0,244 | **0,203** |
| Macro F1 | 0,440 | **0,613** |
| Time-to-Breach | **22,94** | 23,62 |

Klasifikasi terhambat cukup parah — 39% lebih buruk. Menariknya **TTB justru terbantu** oleh multi-task, masuk akal karena memperkirakan "berapa menit lagi jebol" tertolong oleh pemahaman soal suhu masa depan dan jenis kerusakan.

### Perbaikan: suplai statistik ringkasan

Diagnosisnya: XGBoost (baseline pembanding) **menerima** statistik ringkasan secara cuma-cuma, sementara GRU harus **menemukan sendiri** cara menghitungnya dari data mentah — sambil melayani tiga head. Statistik itu lalu disuplai langsung ke head.

| | Macro F1 | Forecast | TTB | PR-AUC |
|---|---|---|---|---|
| Multi-task biasa | 0,440 | 0,244 | 22,94 | 0,591 |
| **Multi-task + statistik** | **0,598** | **0,209** | **21,84** | **0,689** |
| Single-task | 0,613 | 0,203 | 23,62 | 0,679 |

Kerugian multi-task **nyaris hilang** — 0,598 hampir menyamai 0,613 milik single-task, padahal satu model mengerjakan tiga tugas sekaligus. Tambahan parameternya hanya 792.

Ini penting untuk penerapan: backend cukup memuat satu berkas, bukan tiga.

## 5. Ablation B — fine-tuning vs dilatih dari nol

Dua konfigurasi dilatih dengan resep identik: satu memuat bobot GRU hasil prapelatihan, satu dari nol.

| | Fine-tuned | Dari nol |
|---|---|---|
| Loss latih akhir | **2,99** | 3,58 |
| Loss validasi akhir | 4,18 | **2,78** |

Perhatikan arahnya berlawanan: varian pretrained **lebih baik pada data latih** tetapi **lebih buruk pada data validasi**. Itu tanda overfitting.

Penyebabnya ada di korpus prapelatihan yang hanya mampu mengisi 4 dari 12 fitur (lihat `02_pretrain.ipynb`). Bobot GRU terspesialisasi pada distribusi masukan yang timpang, dan spesialisasi itu justru menghambat.

**Keputusan: model produksi dilatih dari nol.**

In [ ]:
from IPython.display import Image, display

display(Image(filename="../reports/loss_curves.png"))

## 6. Konfigurasi akhir

| Parameter | Nilai | Alasan |
|---|---|---|
| Epoch | 30 | loss validasi mendatar setelah ~epoch 25 |
| Batch | 256 | — |
| LR backbone | 5e-4 | lebih pelan daripada head |
| LR head | 2e-3 | head mulai dari acak, perlu mengejar |
| Bobot loss | 30 / 1,0 / 8,0 | menyeimbangkan kontribusi ketiga tugas |
| Bobot kelas | akar dari kebalikan frekuensi | koreksi penuh membuat model justru mengabaikan kelas mayoritas |
| Penjadwal LR | Cosine annealing | — |
| Checkpoint | epoch dengan val terbaik | epoch terakhir belum tentu yang terbaik |

**Catatan tentang bobot kelas.** Percobaan pertama memakai kebalikan frekuensi secara penuh (rasio 1:12). Hasilnya berbalik terlalu jauh — model berhenti menebak `A0` sama sekali dan akurasinya jatuh ke 7%. Akar kuadrat memperlembut rasio menjadi 1:3,5 dan memberi hasil seimbang.

Evaluasi lengkap ada di `04_eval.ipynb`.